# DEMO: Create Your First Dashboard Datasets

This demo is for learners who are brand new to Databricks dashboards.

Your goal in this notebook is to:
* create a new dashboard in the UI
* understand where dashboards live in the workspace
* rename the dashboard
* add all of the prepared datasets that later demos will use

We stay intentionally narrow in scope:
* no visualizations
* no dashboard pages
* no calculated measures
* no advanced formatting or publishing

Before you begin:
1. Run the setup cell below so the demo tables, views, and metric view exist.
2. Keep the dashboard editor open on the **Data** tab for the rest of this demo.
3. Create one new dashboard and add every dataset in this notebook to that same dashboard.

You will add these datasets:
* one SQL dataset built from the star schema
* one Unity Catalog table dataset: `gold_sales`
* three Unity Catalog view datasets: `vw_monthly_revenue`, `vw_region_performance`, `vw_top_products`
* one Unity Catalog metric view dataset: `mv_sales_metrics`

---

In [0]:
%run ./Setup

## Step 1: Create the Dashboard in the UI

Follow these steps in the Databricks workspace:

1. In the left sidebar, click **+ New** and select **Dashboard**.
2. A new draft dashboard opens automatically.
3. Notice the default name. New dashboards start with an auto-generated timestamp-based name.
4. By default, the dashboard is stored in your user workspace folder: `/Workspace/Users/<your-username>`.
5. To confirm where it lives, open **Workspace** in the sidebar and browse to your user folder. Dashboards appear there alongside notebooks, queries, and other workspace assets.
6. You can also find the dashboard from the **Dashboards** listing page in the sidebar.
7. Rename the dashboard now so it is easy to find later. Click the dashboard title at the top of the draft and enter a clear name such as `Demo Dashboard`.

---

## Step 2: Add the Star Schema Dataset with SQL

Use the dashboard UI to create your first dataset:

1. In the dashboard, click the **Data** tab.
2. Click **Add SQL dataset**.
3. If the editor shows **Catalog** and **Schema** selectors, choose the location created by the Setup notebook.
4. Paste in the SQL from the cell below.
5. Click **Run** to confirm the query returns rows.
6. Double click the datasets name to rename the dataset to `star_schema_region_summary`.

---

In [0]:
%sql
-- Dataset 1: custom SQL dataset from the star schema
-- Paste this into the dashboard dataset editor after you click Add SQL dataset.
-- If the query editor shows catalog and schema selectors, point them at the schema
-- created by the Setup notebook before you run this query.

SELECT
  r.region_name,
  r.territory,
  ROUND(SUM(f.net_revenue), 2)          AS total_revenue,
  ROUND(SUM(f.net_revenue - f.cost), 2) AS total_profit,
  COUNT(f.order_id)                     AS order_count
FROM fact_sales f
JOIN dim_region r ON f.region_id = r.region_id
GROUP BY r.region_name, r.territory
ORDER BY total_revenue DESC

## Step 3: Add the Gold Table Dataset from Unity Catalog

This time you will add a dataset by browsing to an existing Unity Catalog table:

1. In the dashboard **Data** tab, click **Add data**.
2. Choose the option to select a **Unity Catalog object**.
3. Browse to the same catalog and schema used by the Setup notebook.
4. Select the table `gold_sales`.
5. The dataset editor opens with a default query, similar to the reference query in the cell above.
6. Rename the dataset to `gold_sales`.
7. Save the dataset.

When you use the "Add data" button and select a table directly, the dashboard automatically creates a **local metric view** (also called a Dashboard Local Metric View / DLMV) rather than a raw SQL dataset. This is the default because metric views are the preferred semantic layer for AI/BI dashboards — they give the system richer metadata about which columns are fields (groupable attributes) and which are measures (aggregatable values), enabling smarter behavior in the widget editor and Genie.

When you manually choose "SQL query" and write `SELECT * FROM gold_sales`, you're explicitly opting into the raw SQL path, which just stores and executes your query as-is with no semantic layer on top.

**Key differences:**

| | Local Metric View | SQL Query Dataset |
| --- | --- | --- |
| Semantic awareness | Fields and measures are declared; the system knows how to aggregate | Flat column list; you handle aggregation in widget expressions |
| Composability | Measures can reference other measures via `MEASURE()` | Derived metrics require custom calculations or inline expressions |
| AI/Genie support | Richer — understands metric intent | Basic — treats columns generically |
| Flexibility | Structured YAML; great for standard analytics | Arbitrary SQL; needed for dynamic params, complex CTEs, or raw extraction |

---

## Step 4: Add the Three View Datasets from Unity Catalog

You will now add three separate datasets, one for each view created during setup:

* `vw_monthly_revenue`
* `vw_region_performance`
* `vw_top_products`

For each view:

1. In the dashboard **Data** tab, click **Add data**.
2. Choose **Unity Catalog object**.
3. Browse to the same catalog and schema.
4. Select one view.
5. Rename the dataset to match the view name.
6. Save it.
7. Repeat the same steps for the next view.

By the end of this step, your dashboard should contain three new view-backed datasets.

---

## Step 5: Add the Metric View Dataset from Unity Catalog

You can add a Unity Catalog metric view directly as a dashboard dataset.

To add the final dataset:

1. In the dashboard **Data** tab, click **Add data**.
2. Choose **Unity Catalog object**.
3. Browse to the same catalog and schema.
4. Select the metric view `mv_sales_metrics`.
5. Rename the dataset to `mv_sales_metrics`.
6. Save the dataset.

Important clarification:
* if you select the metric view directly, the dataset uses the dimensions and measures already defined in the metric view
* if you want to reshape the dataset with SQL or apply filters, then create a SQL dataset instead and use the example in the next cell, where measures are referenced with `MEASURE()`

For this demo, the UI workflow above is valid because the goal is only to add the metric view as a dataset to the dashboard.

---

In [0]:
%sql
-- Optional SQL-based alternative for the metric view dataset
-- Use this only if you want to create the dataset with SQL instead of selecting the
-- metric view directly from Unity Catalog.
-- In that SQL workflow, measures must be referenced with MEASURE().

SELECT
  `Region`,
  `Territory`,
  MEASURE(`Total Revenue`) AS total_revenue,
  MEASURE(`Total Profit`)  AS total_profit,
  MEASURE(`Order Count`)   AS order_count
FROM mv_sales_metrics
GROUP BY ALL
ORDER BY total_revenue DESC

## Final Check: What Should Exist in the Dashboard Now?

Your dashboard should now contain these six datasets:

1. `star_schema_region_summary`
2. `gold_sales`
3. `vw_monthly_revenue`
4. `vw_region_performance`
5. `vw_top_products`
6. `mv_sales_metrics`

Your workspace should also now contain one renamed dashboard in your user folder.

If you need to find it again:
* open **Workspace** and browse to `/Workspace/Users/<your-username>`
* or open the **Dashboards** listing page and search by the name you chose

That is the full scope of this demo: create the dashboard, understand where it lives, rename it, and add the datasets.

---

---

## Summary

By the end of this notebook, the learner should have:

* created a brand new dashboard in the UI
* understood that dashboards are saved as workspace assets in the user's folder by default
* renamed the dashboard so it is easy to find later
* added every required dataset on the dashboard **Data** tab

Nothing else is required here. Later demos can build on these datasets, but this notebook stops once the dashboard and datasets are in place.